In [1]:
import pandas as pd

from src import ProfileReport, Settings
from src import ScoringAgent, TimeSeriesAgent, MetaDataAgent

/usr/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATASET = "./data/Steep_cleaned_cleaned.csv"
df = pd.read_csv(DATASET)

In [3]:
cols = df.columns.to_list()

# Check if we're dealing with timeseries
ts_agent = TimeSeriesAgent()
ts_mode, ts_col = ts_agent.is_timeserie(cols)

if ts_mode:
    df[ts_col] = pd.to_datetime(df[ts_col])


# Sampling
df = df.sample(frac=0.01, random_state=42)

{"ts_mode": true, "timestamp_col": "Timestamp"}


In [4]:
meta_agent = MetaDataAgent()
meta_cols = meta_agent.get_metadata(cols)

{'Timestamp': 'The date and time when the data was recorded.', 'Temps de cycle (s)': 'The total cycle time in seconds for the process.', "Temps d'injection (s)": 'The duration of the injection phase in seconds.', 'Matelas matière (ccm)': 'The material cushion volume in cubic centimeters.', 'Temps de dosage (s)': 'The time taken for dosing in seconds.', 'Commutation postpression Pression injection (bar_spéc)': 'The injection pressure during post-pressure switching, measured in bar.', 'Pression injection Integrale (bar_spez*s)': 'The integral of the injection pressure over time, measured in bar-seconds.', 'Position vis à fin mantien en pression (ccm)': 'The screw position at the end of pressure holding, measured in cubic centimeters.', 'Commutation postpression Position de vis (ccm)': 'The screw position during post-pressure switching, measured in cubic centimeters.', 'BU1 Temp.réel (°C)': 'The real-time temperature of BU1 in degrees Celsius.', 'Pression injection max (bar_spéc)': 'The m

In [13]:
settings = Settings()
settings.html.inline = False  # separate assets instead of inline
settings.html.style.logo = "assets/logo.png"


report = ProfileReport(df,
                       title="Obelix Data Quality Report",
                       progress_bar=True,
                       #
                       explorative=True,
                       minimal=False,
                       sensitive=True,
                       #
                       sortby=ts_col,
                       sort="ascending" if ts_mode else None,
                       tsmode=ts_mode,
                       #
                       variables={"descriptions": meta_cols},
                       #
                       config=settings)

report.to_file("output/report.html")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 12.89it/s]


In [14]:
# scoring
import json
report_json = report.to_json()
report_json = json.loads(report_json)

Render JSON: 100%|██████████| 1/1 [00:00<00:00,  1.68it/s]


In [15]:
scr_agent = ScoringAgent()
score, suggs = scr_agent.get_score(report_json['alerts'])

print(score)
for _ in suggs:
    print("* " + _)

{'score': 65, 'suggs': ["Investigate '[Recul de vis après dosage (ccm)]' for why it has a constant value. This could indicate a sensor issue or a lack of variation in the manufacturing process.", 'Review highly correlated fields like temperatures and other metrics. Excessive correlation can mean redundant information. Consider removing duplicate or unnecessary fields to simplify analysis.', "For imbalanced fields like '[CY5 Temp.réel (°C)]', ensure the data represents a wide range of scenarios. Collect more diverse samples to address imbalance.", "Non-stationary fields like '[BU1 Temp.réel (°C)]' may indicate trends or shifts over time. Check if equipment calibration or external factors are causing these changes.", "For seasonal fields like '[BU1 Temp.réel (°C)]', identify if the seasonality reflects meaningful operational cycles or external influences. If irrelevant, adjust data collection methods.", "Unique values in '[Timestamp]' are expected, but ensure timestamps are accurate and 